# Prompt-only GPT-4o Schema Conformance Experiment

This notebook compares a plain prompt-only GPT-4o extraction baseline against the existing BAML extraction outputs. The purpose is to test whether prompt instruction alone can produce JSON outputs that are structurally valid, ontology-conformant, and ready for Neo4j ingestion.

## Step 1: Setup

Run this cell first. It loads the project paths, OpenAI client, and helper functions used for prompt-only extraction and schema validation.

In [1]:
import base64
import json
import os
import re
import time
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

from openai import OpenAI

load_dotenv()

repo_root = Path.cwd()

if repo_root.name == "knowledge_extraction":
    repo_root = repo_root.parent.parent

DATA_DIR = repo_root / "data" / "processed"
IMAGE_DIR = repo_root / "data" / "raw" / "Manual Book" / "Images"
OUTPUT_DIR = repo_root / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Image dir:", IMAGE_DIR)
print("Output dir:", OUTPUT_DIR)
print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)

Repo root: c:\Users\wrazaq\Documents\Thesis\GraphRAG
Data dir: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\processed
Image dir: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\raw\Manual Book\Images
Output dir: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\processed
OPENAI_API_KEY loaded: True


## Step 2: Check Input Data

This confirms the number of source maintenance-log cases and manual-book images that will be used in the prompt-only experiment.

In [2]:
def count_jsonl(path):
    with open(path, "r", encoding="utf-8") as file:
        return sum(1 for line in file if line.strip())

input_20 = DATA_DIR / "20_cases_for_baml_fewshot.jsonl"
input_remaining = DATA_DIR / "remaining_cases_for_baml_fewshot.jsonl"
manual_images = sorted(IMAGE_DIR.glob("*.jpg")) + sorted(IMAGE_DIR.glob("*.jpeg")) + sorted(IMAGE_DIR.glob("*.png"))

print("20-case evaluation logs:", count_jsonl(input_20))
print("Remaining maintenance logs:", count_jsonl(input_remaining))
print("Manual-book images:", len(manual_images))
print("Manual-book image names:")
for image in manual_images:
    print("-", image.name)

20-case evaluation logs: 20
Remaining maintenance logs: 87
Manual-book images: 11
Manual-book image names:
- Compressor_9600_Brooks-images-0.jpg
- Compressor_9600_Brooks-images-1.jpg
- Cryopump_Brooks_Installation and Maintenance_001.jpg
- Cryopump_Brooks_Installation and Maintenance_002.jpg
- Helix_On-Board controller-2_page-0001.jpg
- Helix_On-Board controller-2_page-0002.jpg
- Ion Beam Drive-images-0.jpg
- Ion Beam Drive-images-1.jpg
- Ion Beam Drive-images-2.jpg
- Ion Beam Drive-images-3.jpg
- Ion Beam Drive-images-4.jpg


Define Schema Validation Logic

In [3]:
EXPECTED_FIELDS_PER_CASE = 6
ENUM_CHECKS_PER_CASE = 2

VALID_MACHINES = {"IBM3", "IBM4"}
VALID_RESOLUTION = {"Resolved", "Unknown", "N/A"}

EXPECTED_TOP_KEYS = {
    "fault_location",
    "fault_symptoms",
    "fault_reason",
    "fault_measures",
    "resolution_status",
}

EXPECTED_LOCATION_KEYS = {"name", "machine"}


def analyze_schema(result):
    missing = 0
    extra = 0
    type_error = 0
    enum_violation = 0

    if not isinstance(result, dict):
        return EXPECTED_FIELDS_PER_CASE, 0, EXPECTED_FIELDS_PER_CASE, ENUM_CHECKS_PER_CASE

    for key in EXPECTED_TOP_KEYS:
        if key not in result:
            missing += 1

    for key in result:
        if key not in EXPECTED_TOP_KEYS:
            extra += 1

    location = result.get("fault_location")

    if not isinstance(location, dict):
        missing += 2
        type_error += 2
        enum_violation += 1
    else:
        if "name" not in location:
            missing += 1
        elif not isinstance(location["name"], str):
            type_error += 1

        if "machine" not in location:
            missing += 1
        else:
            machine = location["machine"]

            if machine is not None and not isinstance(machine, str):
                type_error += 1
            elif machine is not None and machine not in VALID_MACHINES:
                enum_violation += 1

        for key in location:
            if key not in EXPECTED_LOCATION_KEYS:
                extra += 1

    symptoms = result.get("fault_symptoms")

    if symptoms is None:
        missing += 1
    elif not isinstance(symptoms, list):
        type_error += 1
    else:
        for symptom in symptoms:
            if not isinstance(symptom, str):
                type_error += 1

    reasons = result.get("fault_reason")

    if reasons is None:
        missing += 1
    elif not isinstance(reasons, list):
        type_error += 1
    else:
        for reason in reasons:
            if not isinstance(reason, dict):
                type_error += 1
            elif "name" not in reason or not isinstance(reason["name"], str):
                type_error += 1

            if isinstance(reason, dict):
                for key in reason:
                    if key != "name":
                        extra += 1

    measures = result.get("fault_measures")

    if measures is None:
        missing += 1
    elif not isinstance(measures, list):
        type_error += 1
    else:
        for measure in measures:
            if not isinstance(measure, dict):
                type_error += 1
            elif "description" not in measure or not isinstance(measure["description"], str):
                type_error += 1

            if isinstance(measure, dict):
                for key in measure:
                    if key != "description":
                        extra += 1

    resolution = result.get("resolution_status")

    if resolution is None:
        missing += 1
    elif not isinstance(resolution, str):
        type_error += 1
    elif resolution not in VALID_RESOLUTION:
        enum_violation += 1

    return missing, extra, type_error, enum_violation


def is_neo4j_ingestable(report):
    missing, extra, type_error, enum_violation = analyze_schema(report)

    if missing or extra or type_error or enum_violation:
        return False

    location = report.get("fault_location", {})
    name = location.get("name")

    return isinstance(name, str) and bool(name.strip())

Define Prompt-only GPT-4o Extraction Functions

In [5]:
MODEL = "gpt-4o"


def load_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    return rows


def case_to_log_text(case):
    if isinstance(case, str):
        return case

    return case.get("text", "")


def case_id_from_log(log_text, fallback):
    first_line = log_text.splitlines()[0].strip() if log_text else ""

    if first_line.startswith("Case-ID:"):
        return first_line.replace("Case-ID:", "").strip()

    return fallback


def clean_json_text(text):
    if text is None:
        return ""

    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    return text


def parse_json_or_none(text):
    try:
        return json.loads(clean_json_text(text))
    except Exception:
        return None


def build_log_prompt(log_text):
    return f"""
You are an expert in analyzing technical maintenance logs.

Extract the information from this maintenance log.
Do not add explanations, markdown, code fences, comments, or extra fields.

Return the answer in JSON format.

The JSON should contain:
- fault_location: an object containing the component name and the machine name if mentioned.
- fault_symptoms: a list of text strings describing observable problems.
- fault_reason: a list of objects, where each object contains the stated cause of the fault.
- fault_measures: a list of objects, where each object contains a maintenance action or corrective measure.
- resolution_status: a text value indicating whether the fault was resolved, unknown, or not applicable.

---

{log_text}
---
""".strip()


def build_manual_prompt(source_image):
    return """
You are an expert in industrial machine maintenance and repair.

The image is a troubleshooting table from a machine manual.
Extract all distinct fault cases and return only valid JSON.
Do not add explanations, markdown, code fences, comments, or extra fields.

Return exactly one JSON array. Each item must follow this schema:
{
  "fault_location": {"name": "string", "machine": "IBM3" or "IBM4" or null},
  "fault_symptoms": ["string"],
  "fault_reason": [{"name": "string"}],
  "fault_measures": [{"description": "string"}],
  "resolution_status": "Resolved" or "Unknown" or "N/A",
  "source_image": "source_image.jpg"
}

""".strip()


def image_to_data_url(path):
    suffix = path.suffix.lower()
    mime = "image/png" if suffix == ".png" else "image/jpeg"

    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")

    return f"data:{mime};base64,{encoded}"


def call_gpt4o(messages, max_retries=3):
    last_error = None

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                temperature=0,
                messages=messages,
            )

            return response.choices[0].message.content

        except Exception as exc:
            last_error = exc
            wait_time = 2 ** attempt
            print(f"Error: {exc}. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)

    raise last_error

Run Prompt-only Extraction on 20 Evaluation Cases

In [7]:
def extract_logs(input_filename, output_filename):
    cases = load_jsonl(DATA_DIR / input_filename)
    output_path = OUTPUT_DIR / output_filename

    if output_path.exists():
        results = json.loads(output_path.read_text(encoding="utf-8"))
        print(f"Resuming from existing file: {len(results)} already processed")
    else:
        results = []

    for idx, case in enumerate(cases[len(results):], start=len(results) + 1):
        log_text = case_to_log_text(case)
        case_id = case_id_from_log(log_text, f"case_{idx}")

        print(f"Processing {idx}/{len(cases)}: {case_id}")

        try:
            raw_output = call_gpt4o([
                {
                    "role": "system",
                    "content": "You extract structured JSON from maintenance logs."
                },
                {
                    "role": "user",
                    "content": build_log_prompt(log_text)
                }
            ])

            parsed_output = parse_json_or_none(raw_output)

            results.append({
                "case_id": case_id,
                "result": parsed_output
            })

        except Exception as exc:
            print("Error:", exc)

            results.append({
                "case_id": case_id,
                "result": None
            })

        output_path.write_text(
            json.dumps(results, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

    return output_path

In [13]:
prompt_20_path = extract_logs(
    input_filename="20_cases_for_baml_fewshot.jsonl",
    output_filename="prompt_only_gpt4o_20_cases_plain.json",
)

print("Saved:", prompt_20_path)

Processing 1/20: IBM3_C15_23-Feb-15_23-Feb-15
Processing 2/20: IBM3_C26_21-Sep-16_22-Sep-16
Processing 3/20: IBM3_C29_05-Jul-17_12-Jul-17
Processing 4/20: IBM3_C38_18-Nov-19_18-Nov-19
Processing 5/20: IBM3_C45_26-Oct-20_26-Oct-20
Processing 6/20: IBM3_C46_26-Oct-20_26-Oct-20
Processing 7/20: IBM3_C4_16-Oct-13_16-Oct-13
Processing 8/20: IBM3_C53_31-Mar-21_31-Mar-21
Processing 9/20: IBM3_C55_10-Aug-21_10-Aug-21
Processing 10/20: IBM3_C60_29_Mar_22-15-Apr-22
Processing 11/20: IBM3_C74_12-Sep-23_12-Sep-23
Processing 12/20: IBM3_C9_28-Jul-14_28-Jul-14
Processing 13/20: IBM4_C31_27-Mar-17_27-Mar-17
Processing 14/20: IBM4_C37_25-Jun-18_25-Jun-18
Processing 15/20: IBM4_C59_05-Aug-21_05-Aug-21
Processing 16/20: IBM4_C62_23-Dec-21_23-Dec-21
Processing 17/20: IBM4_C74_01-Dec-22_01-Dec-22
Processing 18/20: IBM4_C77_18-Mar-23_19-Apr-23
Processing 19/20: IBM4_C79_13-Jun-23_13-Jun-23
Processing 20/20: IBM4_C91_23-Sept-24_01-Oct-24
Saved: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\processed\prompt

schema evaluator

In [18]:
import json
from pathlib import Path

repo_root = Path.cwd()

if repo_root.name == "knowledge_extraction":
    repo_root = repo_root.parent.parent

prompt_file = repo_root / "data" / "processed" / "prompt_only_gpt4o_20_cases_plain.json"

print("File exists:", prompt_file.exists())
print("File path:", prompt_file)

File exists: True
File path: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\processed\prompt_only_gpt4o_20_cases_plain.json


In [16]:
EXPECTED_FIELDS_PER_CASE = 6
ENUM_CHECKS_PER_CASE = 2

VALID_MACHINES = {"IBM3", "IBM4"}
VALID_RESOLUTION = {"Resolved", "Unknown", "N/A"}

EXPECTED_TOP_KEYS = {
    "fault_location",
    "fault_symptoms",
    "fault_reason",
    "fault_measures",
    "resolution_status",
}

EXPECTED_LOCATION_KEYS = {"name", "machine"}


def analyze_schema(result):
    missing = 0
    extra = 0
    type_error = 0
    enum_violation = 0

    if not isinstance(result, dict):
        return EXPECTED_FIELDS_PER_CASE, 0, EXPECTED_FIELDS_PER_CASE, ENUM_CHECKS_PER_CASE

    for key in EXPECTED_TOP_KEYS:
        if key not in result:
            missing += 1

    for key in result:
        if key not in EXPECTED_TOP_KEYS:
            extra += 1

    location = result.get("fault_location")

    if not isinstance(location, dict):
        missing += 2
        type_error += 2
        enum_violation += 1
    else:
        if "name" not in location:
            missing += 1
        elif not isinstance(location["name"], str):
            type_error += 1

        if "machine" not in location:
            missing += 1
        else:
            machine = location["machine"]
            if machine is not None and not isinstance(machine, str):
                type_error += 1
            elif machine is not None and machine not in VALID_MACHINES:
                enum_violation += 1

        for key in location:
            if key not in EXPECTED_LOCATION_KEYS:
                extra += 1

    symptoms = result.get("fault_symptoms")

    if symptoms is None:
        missing += 1
    elif not isinstance(symptoms, list):
        type_error += 1
    else:
        for symptom in symptoms:
            if not isinstance(symptom, str):
                type_error += 1

    reasons = result.get("fault_reason")

    if reasons is None:
        missing += 1
    elif not isinstance(reasons, list):
        type_error += 1
    else:
        for reason in reasons:
            if not isinstance(reason, dict):
                type_error += 1
            elif "name" not in reason or not isinstance(reason["name"], str):
                type_error += 1

            if isinstance(reason, dict):
                for key in reason:
                    if key != "name":
                        extra += 1

    measures = result.get("fault_measures")

    if measures is None:
        missing += 1
    elif not isinstance(measures, list):
        type_error += 1
    else:
        for measure in measures:
            if not isinstance(measure, dict):
                type_error += 1
            elif "description" not in measure or not isinstance(measure["description"], str):
                type_error += 1

            if isinstance(measure, dict):
                for key in measure:
                    if key != "description":
                        extra += 1

    resolution = result.get("resolution_status")

    if resolution is None:
        missing += 1
    elif not isinstance(resolution, str):
        type_error += 1
    elif resolution not in VALID_RESOLUTION:
        enum_violation += 1

    return missing, extra, type_error, enum_violation

In [20]:
data = json.loads(prompt_file.read_text(encoding="utf-8"))

total_reports = len(data)
valid_cases = 0
json_parse_failures = 0

total_missing = 0
total_extra = 0
total_type_error = 0
total_enum_violation = 0

invalid_examples = []

for item in data:
    result = item.get("result")

    if result is None:
        json_parse_failures += 1

    missing, extra, type_error, enum_violation = analyze_schema(result)

    total_missing += missing
    total_extra += extra
    total_type_error += type_error
    total_enum_violation += enum_violation

    if missing == 0 and extra == 0 and type_error == 0 and enum_violation == 0:
        valid_cases += 1
    else:
        invalid_examples.append({
            "case_id": item.get("case_id"),
            "missing": missing,
            "extra": extra,
            "type_error": type_error,
            "enum_violation": enum_violation,
            "result": result,
            "raw_response": item.get("raw_response"),
        })

print("Prompt-only GPT-4o schema conformance")
print("=" * 60)
print(f"Total reports: {total_reports}")
print(f"JSON parse failures: {json_parse_failures}")
print(f"Valid cases: {valid_cases} / {total_reports}")
print(f"Missing fields: {total_missing} / {total_reports * EXPECTED_FIELDS_PER_CASE}")
print(f"Extra fields: {total_extra} / {total_reports * EXPECTED_FIELDS_PER_CASE}")
print(f"Type errors: {total_type_error} / {total_reports * EXPECTED_FIELDS_PER_CASE}")
print(f"Enumeration violations: {total_enum_violation} / {total_reports * ENUM_CHECKS_PER_CASE}")

Prompt-only GPT-4o schema conformance
Total reports: 20
JSON parse failures: 0
Valid cases: 20 / 20
Missing fields: 0 / 120
Extra fields: 0 / 120
Type errors: 0 / 120
Enumeration violations: 0 / 40


SETUP IMAGE EXTRACTION

In [21]:
import base64
import json
import os
import re
import time
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

from openai import OpenAI

load_dotenv()

repo_root = Path.cwd()

if repo_root.name == "knowledge_extraction":
    repo_root = repo_root.parent.parent

IMAGE_DIR = repo_root / "data" / "raw" / "Manual Book" / "Images"
OUTPUT_DIR = repo_root / "experiments" / "prompt_only_gpt4o"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Image dir:", IMAGE_DIR)
print("Output dir:", OUTPUT_DIR)
print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)

Image dir: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\raw\Manual Book\Images
Output dir: c:\Users\wrazaq\Documents\Thesis\GraphRAG\experiments\prompt_only_gpt4o
OPENAI_API_KEY loaded: True


In [22]:
image_paths = (
    sorted(IMAGE_DIR.glob("*.jpg")) +
    sorted(IMAGE_DIR.glob("*.jpeg")) +
    sorted(IMAGE_DIR.glob("*.png"))
)

print("Number of images:", len(image_paths))

for image_path in image_paths:
    print(image_path.name)

Number of images: 11
Compressor_9600_Brooks-images-0.jpg
Compressor_9600_Brooks-images-1.jpg
Cryopump_Brooks_Installation and Maintenance_001.jpg
Cryopump_Brooks_Installation and Maintenance_002.jpg
Helix_On-Board controller-2_page-0001.jpg
Helix_On-Board controller-2_page-0002.jpg
Ion Beam Drive-images-0.jpg
Ion Beam Drive-images-1.jpg
Ion Beam Drive-images-2.jpg
Ion Beam Drive-images-3.jpg
Ion Beam Drive-images-4.jpg


Define Image Helper Functions

In [23]:
MODEL = "gpt-4o"


def image_to_data_url(path):
    suffix = path.suffix.lower()
    mime = "image/png" if suffix == ".png" else "image/jpeg"

    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")

    return f"data:{mime};base64,{encoded}"


def clean_json_text(text):
    if text is None:
        return ""

    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    return text


def parse_json_or_none(text):
    try:
        return json.loads(clean_json_text(text))
    except Exception:
        return None


def call_gpt4o_image(messages, max_retries=3):
    last_error = None

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                temperature=0,
                messages=messages,
            )

            return response.choices[0].message.content

        except Exception as exc:
            last_error = exc
            wait_time = 2 ** attempt
            print(f"Error: {exc}. Retrying in {wait_time} seconds...")
            time.sleep(wait_time)

    raise last_error

Test on One Image First

In [28]:
test_image = image_paths[0]

print("Testing image:", test_image.name)

raw_output = call_gpt4o_image([
    {
        "role": "system",
        "content": "You extract structured JSON from troubleshooting-table images."
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": build_manual_prompt(test_image.name)
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": image_to_data_url(test_image)
                }
            }
        ]
    }
])

parsed_output = parse_json_or_none(raw_output)

print("Raw output:")
print(raw_output)

print("\nParsed output:")
print(json.dumps(parsed_output, indent=2, ensure_ascii=False))

Testing image: Compressor_9600_Brooks-images-0.jpg
Raw output:
[
  {
    "fault_location": {"name": "System circuit breaker (CB1)", "machine": null},
    "fault_symptoms": ["Trips immediately to the OFF (0) position when switched to the ON (1) position."],
    "fault_reason": [{"name": "Incorrect phasing of input power."}],
    "fault_measures": [{"description": "Check phasing of input power. Refer to 'Phase Check' in 'Section 3 - Installation'."}],
    "resolution_status": "Unknown",
    "source_image": "source_image.jpg"
  },
  {
    "fault_location": {"name": "System (CB1) and Control Circuit (CB2)", "machine": null},
    "fault_symptoms": ["Circuit breakers remain in the ON (1) position when switched ON but the compressor will not run."],
    "fault_reason": [{"name": "No power coming from source."}],
    "fault_measures": [{"description": "Check source fuses, circuit breakers, and wiring associated with the power source. Repair as needed."}],
    "resolution_status": "Unknown",
  

Save the One-image Test Output

In [29]:
test_output_path = OUTPUT_DIR / "prompt_only_gpt4o_manual_image_test.json"

test_result = {
    "source_image": test_image.name,
    "result": parsed_output,
    "raw_response": raw_output,
}

test_output_path.write_text(
    json.dumps(test_result, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Saved:", test_output_path)

Saved: c:\Users\wrazaq\Documents\Thesis\GraphRAG\experiments\prompt_only_gpt4o\prompt_only_gpt4o_manual_image_test.json


evaluate 1 image

In [30]:
import json
from pathlib import Path

one_image_file = Path(r"C:\Users\wrazaq\.codex\attachments\cb6922e4-0c48-4fc5-8d5d-396f880627e5\pasted-text.txt")

with open(one_image_file, "r", encoding="utf-8") as file:
    one_image_data = json.load(file)

print(type(one_image_data))
print(one_image_data.keys())
print("Source image:", one_image_data.get("source_image"))
print("Number of extracted reports:", len(one_image_data.get("result", [])))

<class 'dict'>
dict_keys(['source_image', 'result', 'raw_response'])
Source image: Compressor_9600_Brooks-images-0.jpg
Number of extracted reports: 5


In [31]:
EXPECTED_FIELDS_PER_CASE = 6
ENUM_CHECKS_PER_CASE = 2

VALID_MACHINES = {"IBM3", "IBM4"}
VALID_RESOLUTION = {"Resolved", "Unknown", "N/A"}

EXPECTED_TOP_KEYS = {
    "fault_location",
    "fault_symptoms",
    "fault_reason",
    "fault_measures",
    "resolution_status",
    "source_image",
}

EXPECTED_LOCATION_KEYS = {"name", "machine"}


def analyze_schema(result):
    missing = 0
    extra = 0
    type_error = 0
    enum_violation = 0

    if not isinstance(result, dict):
        return EXPECTED_FIELDS_PER_CASE, 0, EXPECTED_FIELDS_PER_CASE, ENUM_CHECKS_PER_CASE

    for key in EXPECTED_TOP_KEYS:
        if key not in result:
            missing += 1

    for key in result:
        if key not in EXPECTED_TOP_KEYS:
            extra += 1

    location = result.get("fault_location")

    if not isinstance(location, dict):
        missing += 2
        type_error += 2
        enum_violation += 1
    else:
        if "name" not in location:
            missing += 1
        elif not isinstance(location["name"], str):
            type_error += 1

        if "machine" not in location:
            missing += 1
        else:
            machine = location["machine"]

            if machine is not None and not isinstance(machine, str):
                type_error += 1
            elif machine is not None and machine not in VALID_MACHINES:
                enum_violation += 1

        for key in location:
            if key not in EXPECTED_LOCATION_KEYS:
                extra += 1

    symptoms = result.get("fault_symptoms")

    if symptoms is None:
        missing += 1
    elif not isinstance(symptoms, list):
        type_error += 1
    else:
        for symptom in symptoms:
            if not isinstance(symptom, str):
                type_error += 1

    reasons = result.get("fault_reason")

    if reasons is None:
        missing += 1
    elif not isinstance(reasons, list):
        type_error += 1
    else:
        for reason in reasons:
            if not isinstance(reason, dict):
                type_error += 1
            elif "name" not in reason or not isinstance(reason["name"], str):
                type_error += 1

            if isinstance(reason, dict):
                for key in reason:
                    if key != "name":
                        extra += 1

    measures = result.get("fault_measures")

    if measures is None:
        missing += 1
    elif not isinstance(measures, list):
        type_error += 1
    else:
        for measure in measures:
            if not isinstance(measure, dict):
                type_error += 1
            elif "description" not in measure or not isinstance(measure["description"], str):
                type_error += 1

            if isinstance(measure, dict):
                for key in measure:
                    if key != "description":
                        extra += 1

    resolution = result.get("resolution_status")

    if resolution is None:
        missing += 1
    elif not isinstance(resolution, str):
        type_error += 1
    elif resolution not in VALID_RESOLUTION:
        enum_violation += 1

    source_image = result.get("source_image")

    if source_image is None:
        missing += 1
    elif not isinstance(source_image, str):
        type_error += 1

    return missing, extra, type_error, enum_violation

In [32]:
reports = one_image_data.get("result", [])

total_reports = len(reports)
valid_cases = 0

total_missing = 0
total_extra = 0
total_type_error = 0
total_enum_violation = 0

invalid_examples = []

for idx, report in enumerate(reports, start=1):
    missing, extra, type_error, enum_violation = analyze_schema(report)

    total_missing += missing
    total_extra += extra
    total_type_error += type_error
    total_enum_violation += enum_violation

    if missing == 0 and extra == 0 and type_error == 0 and enum_violation == 0:
        valid_cases += 1
    else:
        invalid_examples.append({
            "index": idx,
            "missing": missing,
            "extra": extra,
            "type_error": type_error,
            "enum_violation": enum_violation,
            "report": report,
        })

print("Prompt-only GPT-4o image schema conformance")
print("=" * 60)
print("Source image:", one_image_data.get("source_image"))
print(f"Total extracted reports: {total_reports}")
print(f"Valid cases: {valid_cases} / {total_reports}")
print(f"Missing fields: {total_missing} / {total_reports * EXPECTED_FIELDS_PER_CASE}")
print(f"Extra fields: {total_extra} / {total_reports * EXPECTED_FIELDS_PER_CASE}")
print(f"Type errors: {total_type_error} / {total_reports * EXPECTED_FIELDS_PER_CASE}")
print(f"Enumeration violations: {total_enum_violation} / {total_reports * ENUM_CHECKS_PER_CASE}")

Prompt-only GPT-4o image schema conformance
Source image: Compressor_9600_Brooks-images-0.jpg
Total extracted reports: 5
Valid cases: 5 / 5
Missing fields: 0 / 30
Extra fields: 0 / 30
Type errors: 0 / 30
Enumeration violations: 0 / 10


Terakhir extracted logs and 1 image, dan semua evaluation for schema conformance berhasil 100%. mungkin habis ini harus mengubah prompt nya untuk tidak terlalu detail.

In [23]:
def extract_logs_with_raw_audit(input_filename, output_filename):
    cases = load_jsonl(DATA_DIR / input_filename)
    output_path = OUTPUT_DIR / output_filename

    results = []

    for idx, case in enumerate(cases, start=1):
        log_text = case_to_log_text(case)
        case_id = case_id_from_log(log_text, f"case_{idx}")

        print(f"Processing {idx}/{len(cases)}: {case_id}")

        try:
            raw_output = call_gpt4o([
                {
                    "role": "system",
                    "content": "You extract structured JSON from maintenance logs."
                },
                {
                    "role": "user",
                    "content": build_log_prompt(log_text)
                }
            ])

            # Test 1: can the raw model answer be parsed directly?
            try:
                raw_parsed = json.loads(raw_output)
                raw_json_valid = True
            except Exception:
                raw_parsed = None
                raw_json_valid = False

            # Test 2: can the model answer be parsed after simple cleaning?
            cleaned_parsed = parse_json_or_none(raw_output)
            cleaned_json_valid = cleaned_parsed is not None

            results.append({
                "case_id": case_id,
                "raw_output": raw_output,
                "raw_json_valid": raw_json_valid,
                "cleaned_json_valid": cleaned_json_valid,
                "result": cleaned_parsed
            })

        except Exception as exc:
            print("Error:", exc)

            results.append({
                "case_id": case_id,
                "raw_output": None,
                "raw_json_valid": False,
                "cleaned_json_valid": False,
                "result": None,
                "error": str(exc)
            })

        output_path.write_text(
            json.dumps(results, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )

    return output_path

In [24]:
prompt_20_raw_audit_loose_path = extract_logs_with_raw_audit(
    input_filename="20_cases_for_baml_fewshot.jsonl",
    output_filename="prompt_only_gpt4o_20_cases_raw_audit_loose_prompt.json"
)

print("Saved:", prompt_20_raw_audit_loose_path)

Processing 1/20: IBM3_C15_23-Feb-15_23-Feb-15
Processing 2/20: IBM3_C26_21-Sep-16_22-Sep-16
Processing 3/20: IBM3_C29_05-Jul-17_12-Jul-17
Processing 4/20: IBM3_C38_18-Nov-19_18-Nov-19
Processing 5/20: IBM3_C45_26-Oct-20_26-Oct-20
Processing 6/20: IBM3_C46_26-Oct-20_26-Oct-20
Processing 7/20: IBM3_C4_16-Oct-13_16-Oct-13
Processing 8/20: IBM3_C53_31-Mar-21_31-Mar-21
Processing 9/20: IBM3_C55_10-Aug-21_10-Aug-21
Processing 10/20: IBM3_C60_29_Mar_22-15-Apr-22
Processing 11/20: IBM3_C74_12-Sep-23_12-Sep-23
Processing 12/20: IBM3_C9_28-Jul-14_28-Jul-14
Processing 13/20: IBM4_C31_27-Mar-17_27-Mar-17
Processing 14/20: IBM4_C37_25-Jun-18_25-Jun-18
Processing 15/20: IBM4_C59_05-Aug-21_05-Aug-21
Processing 16/20: IBM4_C62_23-Dec-21_23-Dec-21
Processing 17/20: IBM4_C74_01-Dec-22_01-Dec-22
Processing 18/20: IBM4_C77_18-Mar-23_19-Apr-23
Processing 19/20: IBM4_C79_13-Jun-23_13-Jun-23
Processing 20/20: IBM4_C91_23-Sept-24_01-Oct-24
Saved: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\processed\prompt

In [4]:
loose_audit_data = json.loads(
    prompt_20_raw_audit_loose_path.read_text(encoding="utf-8")
)

print("Total cases:", len(loose_audit_data))

NameError: name 'prompt_20_raw_audit_loose_path' is not defined

In [26]:
total_cases = len(loose_audit_data)

raw_valid = sum(item["raw_json_valid"] for item in loose_audit_data)
cleaned_valid = sum(item["cleaned_json_valid"] for item in loose_audit_data)

print("Raw JSON valid:", raw_valid, "/", total_cases)
print("Cleaned JSON valid:", cleaned_valid, "/", total_cases)

Raw JSON valid: 20 / 20
Cleaned JSON valid: 20 / 20


In [27]:
raw_valid_reports = []

for item in loose_audit_data:
    if item["raw_json_valid"]:
        raw_valid_reports.append(json.loads(item["raw_output"]))

total_missing = 0
total_extra = 0
total_type_error = 0
total_enum_violation = 0
valid_schema_cases = 0

for report in raw_valid_reports:
    missing, extra, type_error, enum_violation = analyze_schema(report)

    total_missing += missing
    total_extra += extra
    total_type_error += type_error
    total_enum_violation += enum_violation

    if missing == 0 and extra == 0 and type_error == 0 and enum_violation == 0:
        valid_schema_cases += 1

n = len(raw_valid_reports)

print("Raw-valid reports:", n)
print("Schema-valid among raw-valid reports:", valid_schema_cases, "/", n)
print("Missing fields:", total_missing, "/", n * EXPECTED_FIELDS_PER_CASE)
print("Extra fields:", total_extra, "/", n * EXPECTED_FIELDS_PER_CASE)
print("Type errors:", total_type_error, "/", n * EXPECTED_FIELDS_PER_CASE)
print("Enum violations:", total_enum_violation, "/", n * ENUM_CHECKS_PER_CASE)

Raw-valid reports: 20
Schema-valid among raw-valid reports: 20 / 20
Missing fields: 0 / 120
Extra fields: 0 / 120
Type errors: 0 / 120
Enum violations: 0 / 40


In [11]:
loose_cases = load_jsonl(DATA_DIR / "20_cases_for_baml_fewshot.jsonl")

loose_output_path = DATA_DIR / "prompt_only_gpt4o_20_cases_loose_one_by_one_raw_audit.json"

if loose_output_path.exists():
    loose_results = json.loads(loose_output_path.read_text(encoding="utf-8"))
    print("Existing extracted cases:", len(loose_results))
else:
    loose_results = []
    loose_output_path.write_text(
        json.dumps(loose_results, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print("Created new file:", loose_output_path)

print("Total cases to extract:", len(loose_cases))

Created new file: c:\Users\wrazaq\Documents\Thesis\GraphRAG\data\processed\prompt_only_gpt4o_20_cases_loose_one_by_one_raw_audit.json
Total cases to extract: 20


In [8]:
def extract_one_loose_case(case_no):
    """
    case_no starts from 1.
    Example: extract_one_loose_case(1)
    """

    if case_no < 1 or case_no > len(loose_cases):
        raise ValueError(f"case_no must be between 1 and {len(loose_cases)}")

    case = loose_cases[case_no - 1]
    log_text = case_to_log_text(case)
    case_id = case_id_from_log(log_text, f"case_{case_no}")

    loose_results = json.loads(loose_output_path.read_text(encoding="utf-8"))

    existing_ids = {item["case_id"] for item in loose_results}

    if case_id in existing_ids:
        print("Already extracted:", case_id)
        return case_id

    print(f"Extracting case {case_no}/{len(loose_cases)}: {case_id}")

    raw_output = call_gpt4o([
        {
            "role": "system",
            "content": "You extract fault information from maintenance logs."
        },
        {
            "role": "user",
            "content": build_log_prompt(log_text)
        }
    ])

    # Raw JSON validity: no cleaning, no repair.
    try:
        raw_parsed = json.loads(raw_output)
        raw_json_valid = True
    except Exception:
        raw_parsed = None
        raw_json_valid = False

    # Cleaned JSON validity: allows simple removal of code fences.
    cleaned_parsed = parse_json_or_none(raw_output)
    cleaned_json_valid = cleaned_parsed is not None

    record = {
        "case_no": case_no,
        "case_id": case_id,
        "raw_output": raw_output,
        "raw_json_valid": raw_json_valid,
        "cleaned_json_valid": cleaned_json_valid,
        "result": cleaned_parsed
    }

    loose_results.append(record)

    loose_output_path.write_text(
        json.dumps(loose_results, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print("Saved:", case_id)
    print("Raw JSON valid:", raw_json_valid)
    print("Cleaned JSON valid:", cleaned_json_valid)

    return case_id

In [9]:
def evaluate_one_loose_case(case_id):
    loose_results = json.loads(loose_output_path.read_text(encoding="utf-8"))

    matches = [
        item for item in loose_results
        if item["case_id"] == case_id
    ]

    if not matches:
        raise ValueError(f"Case not found: {case_id}")

    item = matches[0]

    print("=" * 100)
    print("Case:", item["case_id"])
    print("=" * 100)

    print("Raw JSON valid:", item["raw_json_valid"])
    print("Cleaned JSON valid:", item["cleaned_json_valid"])

    result = item["result"]

    missing, extra, type_error, enum_violation = analyze_schema(result)

    schema_valid = (
        missing == 0
        and extra == 0
        and type_error == 0
        and enum_violation == 0
    )

    print("Schema valid after cleaning:", schema_valid)
    print("Neo4j ingestable:", is_neo4j_ingestable(result))
    print()
    print("Missing fields:", missing)
    print("Extra fields:", extra)
    print("Type errors:", type_error)
    print("Enum violations:", enum_violation)
    print()
    print("Raw output:")
    print(item["raw_output"])
    print()
    print("Parsed result:")
    print(json.dumps(result, indent=2, ensure_ascii=False))

In [12]:
case_id = extract_one_loose_case(1)
evaluate_one_loose_case(case_id)

Extracting case 1/20: IBM3_C15_23-Feb-15_23-Feb-15
Saved: IBM3_C15_23-Feb-15_23-Feb-15
Raw JSON valid: True
Cleaned JSON valid: True
Case: IBM3_C15_23-Feb-15_23-Feb-15
Raw JSON valid: True
Cleaned JSON valid: True
Schema valid after cleaning: False
Neo4j ingestable: False

Missing fields: 2
Extra fields: 2
Type errors: 0
Enum violations: 0

Raw output:
{
  "fault_location": {
    "component_name": "cryocompressor",
    "machine_name": "9600 compressor"
  },
  "fault_symptoms": [
    "He-verlies/tekort in cryocompressor",
    "Displacer maakt een wat hoog piepend/schrapend geluid",
    "Temp. Is 11K"
  ],
  "fault_reason": [],
  "fault_measures": [],
  "resolution_status": "Unknown"
}

Parsed result:
{
  "fault_location": {
    "component_name": "cryocompressor",
    "machine_name": "9600 compressor"
  },
  "fault_symptoms": [
    "He-verlies/tekort in cryocompressor",
    "Displacer maakt een wat hoog piepend/schrapend geluid",
    "Temp. Is 11K"
  ],
  "fault_reason": [],
  "fault_mea

In [13]:
case_id = extract_one_loose_case(2)
evaluate_one_loose_case(case_id)

Extracting case 2/20: IBM3_C26_21-Sep-16_22-Sep-16
Saved: IBM3_C26_21-Sep-16_22-Sep-16
Raw JSON valid: True
Cleaned JSON valid: True
Case: IBM3_C26_21-Sep-16_22-Sep-16
Raw JSON valid: True
Cleaned JSON valid: True
Schema valid after cleaning: False
Neo4j ingestable: False

Missing fields: 2
Extra fields: 5
Type errors: 3
Enum violations: 0

Raw output:
{
  "fault_location": {
    "component_name": "Armkanteling",
    "machine_name": null
  },
  "fault_symptoms": [
    "Armkanteling werkt niet (voeding wel aan)",
    "Shutterpositie indicatie: tussenpositie"
  ],
  "fault_reason": [
    {
      "cause": "Shutterpositie in tussenpositie"
    }
  ],
  "fault_measures": [
    {
      "action": "JP6 los en pen 2 en 3 aan elkaar gesoldeerd"
    },
    {
      "action": "Eindschakelaar(s) zijn met jumper overbrugd"
    }
  ],
  "resolution_status": "Unknown"
}

Parsed result:
{
  "fault_location": {
    "component_name": "Armkanteling",
    "machine_name": null
  },
  "fault_symptoms": [
    

In [14]:
case_id = extract_one_loose_case(3)
evaluate_one_loose_case(case_id)

Extracting case 3/20: IBM3_C29_05-Jul-17_12-Jul-17
Saved: IBM3_C29_05-Jul-17_12-Jul-17
Raw JSON valid: True
Cleaned JSON valid: True
Case: IBM3_C29_05-Jul-17_12-Jul-17
Raw JSON valid: True
Cleaned JSON valid: True
Schema valid after cleaning: False
Neo4j ingestable: False

Missing fields: 2
Extra fields: 5
Type errors: 3
Enum violations: 0

Raw output:
{
  "fault_location": {
    "component_name": "tafel",
    "machine_name": "IBM3"
  },
  "fault_symptoms": [
    "probleem met tafel",
    "geen positie / rpm detectie",
    "proces start niet",
    "lichtsluisje van schijf van / onder tafel deed het niet"
  ],
  "fault_reason": [
    {
      "cause": "lichtsluisje van schijf van / onder tafel deed het niet"
    }
  ],
  "fault_measures": [
    {
      "action": "schoongemaakt"
    },
    {
      "action": "blauwe draad aangesoldeerd"
    }
  ],
  "resolution_status": "Unknown"
}

Parsed result:
{
  "fault_location": {
    "component_name": "tafel",
    "machine_name": "IBM3"
  },
  "fau

In [15]:
case_id = extract_one_loose_case(4)
evaluate_one_loose_case(case_id)

Extracting case 4/20: IBM3_C38_18-Nov-19_18-Nov-19
Saved: IBM3_C38_18-Nov-19_18-Nov-19
Raw JSON valid: True
Cleaned JSON valid: True
Case: IBM3_C38_18-Nov-19_18-Nov-19
Raw JSON valid: True
Cleaned JSON valid: True
Schema valid after cleaning: False
Neo4j ingestable: False

Missing fields: 2
Extra fields: 4
Type errors: 2
Enum violations: 0

Raw output:
{
  "fault_location": {
    "component_name": "rotatietafel",
    "machine_name": "ibm"
  },
  "fault_symptoms": [
    "Nulpositie vinden lukte niet",
    "Homing lukt niet"
  ],
  "fault_reason": [
    {
      "cause": "voeding PM150 was uitgezet"
    }
  ],
  "fault_measures": [
    {
      "action": "Voeding ibm uit, weer aan"
    }
  ],
  "resolution_status": "Resolved"
}

Parsed result:
{
  "fault_location": {
    "component_name": "rotatietafel",
    "machine_name": "ibm"
  },
  "fault_symptoms": [
    "Nulpositie vinden lukte niet",
    "Homing lukt niet"
  ],
  "fault_reason": [
    {
      "cause": "voeding PM150 was uitgezet"
 

In [16]:
case_id = extract_one_loose_case(5)
evaluate_one_loose_case(case_id)

Extracting case 5/20: IBM3_C45_26-Oct-20_26-Oct-20
Saved: IBM3_C45_26-Oct-20_26-Oct-20
Raw JSON valid: True
Cleaned JSON valid: True
Case: IBM3_C45_26-Oct-20_26-Oct-20
Raw JSON valid: True
Cleaned JSON valid: True
Schema valid after cleaning: False
Neo4j ingestable: False

Missing fields: 2
Extra fields: 4
Type errors: 2
Enum violations: 0

Raw output:
{
  "fault_location": {
    "component_name": "Acc.grid",
    "machine_name": null
  },
  "fault_symptoms": [
    "sluiting tegen massa",
    "Doorslag naar de \"deksel\""
  ],
  "fault_reason": [
    {
      "cause": "keramiek van schoefjes die grid-set bij elkaar houden"
    }
  ],
  "fault_measures": [
    {
      "action": "schoefjes vervangen door nieuwe met schone keramiek"
    }
  ],
  "resolution_status": "N/A"
}

Parsed result:
{
  "fault_location": {
    "component_name": "Acc.grid",
    "machine_name": null
  },
  "fault_symptoms": [
    "sluiting tegen massa",
    "Doorslag naar de \"deksel\""
  ],
  "fault_reason": [
    {
 

In [17]:
case_id = extract_one_loose_case(6)
evaluate_one_loose_case(case_id)

Extracting case 6/20: IBM3_C46_26-Oct-20_26-Oct-20
Saved: IBM3_C46_26-Oct-20_26-Oct-20
Raw JSON valid: True
Cleaned JSON valid: True
Case: IBM3_C46_26-Oct-20_26-Oct-20
Raw JSON valid: True
Cleaned JSON valid: True
Schema valid after cleaning: False
Neo4j ingestable: False

Missing fields: 2
Extra fields: 3
Type errors: 1
Enum violations: 0

Raw output:
{
  "fault_location": {
    "component_name": "cryopomp",
    "machine_name": "IBM3"
  },
  "fault_symptoms": [
    "Uitval cryopomp",
    "Cryopomp temp. was na afloop: 60K",
    "IBM4 zakt niet verder dan 150K"
  ],
  "fault_reason": [
    {
      "cause": "koelwater is onvoldoende om beide comperessoren te koelen"
    }
  ],
  "fault_measures": [],
  "resolution_status": "Unknown"
}

Parsed result:
{
  "fault_location": {
    "component_name": "cryopomp",
    "machine_name": "IBM3"
  },
  "fault_symptoms": [
    "Uitval cryopomp",
    "Cryopomp temp. was na afloop: 60K",
    "IBM4 zakt niet verder dan 150K"
  ],
  "fault_reason": [
  

In [18]:
case_id = extract_one_loose_case(7)
evaluate_one_loose_case(case_id)

Extracting case 7/20: IBM3_C4_16-Oct-13_16-Oct-13
Saved: IBM3_C4_16-Oct-13_16-Oct-13
Raw JSON valid: True
Cleaned JSON valid: True
Case: IBM3_C4_16-Oct-13_16-Oct-13
Raw JSON valid: True
Cleaned JSON valid: True
Schema valid after cleaning: False
Neo4j ingestable: False

Missing fields: 2
Extra fields: 5
Type errors: 3
Enum violations: 0

Raw output:
{
  "fault_location": {
    "component_name": "Stappenmotor controler",
    "machine_name": "PM170"
  },
  "fault_symptoms": [
    "Arm tilt werkt niet"
  ],
  "fault_reason": [
    {
      "cause": "Stappenmotor controler kreeg geen voeding"
    },
    {
      "cause": "Automatische zekering CB4 lag er uit"
    }
  ],
  "fault_measures": [
    {
      "action": "Na inschakelen werkte alles weer"
    }
  ],
  "resolution_status": "Resolved"
}

Parsed result:
{
  "fault_location": {
    "component_name": "Stappenmotor controler",
    "machine_name": "PM170"
  },
  "fault_symptoms": [
    "Arm tilt werkt niet"
  ],
  "fault_reason": [
    {
 

In [19]:
case_id = extract_one_loose_case(8)
evaluate_one_loose_case(case_id)

Extracting case 8/20: IBM3_C53_31-Mar-21_31-Mar-21
Saved: IBM3_C53_31-Mar-21_31-Mar-21
Raw JSON valid: True
Cleaned JSON valid: True
Case: IBM3_C53_31-Mar-21_31-Mar-21
Raw JSON valid: True
Cleaned JSON valid: True
Schema valid after cleaning: False
Neo4j ingestable: False

Missing fields: 2
Extra fields: 5
Type errors: 3
Enum violations: 0

Raw output:
{
  "fault_location": {
    "component_name": "cryopomp",
    "machine_name": "display 13K"
  },
  "fault_symptoms": [
    "storingsmelding tijdens het programma 'pompen'",
    "cryopomp nog niet koud"
  ],
  "fault_reason": [
    {
      "cause": "cryopomp nog niet koud"
    }
  ],
  "fault_measures": [
    {
      "action": "full regeneratie opgestart"
    },
    {
      "action": "RLY op auto modus gezet"
    }
  ],
  "resolution_status": "Resolved"
}

Parsed result:
{
  "fault_location": {
    "component_name": "cryopomp",
    "machine_name": "display 13K"
  },
  "fault_symptoms": [
    "storingsmelding tijdens het programma 'pompen'

In [20]:
case_id = extract_one_loose_case(9)
evaluate_one_loose_case(case_id)

Extracting case 9/20: IBM3_C55_10-Aug-21_10-Aug-21
Saved: IBM3_C55_10-Aug-21_10-Aug-21
Raw JSON valid: True
Cleaned JSON valid: True
Case: IBM3_C55_10-Aug-21_10-Aug-21
Raw JSON valid: True
Cleaned JSON valid: True
Schema valid after cleaning: False
Neo4j ingestable: False

Missing fields: 2
Extra fields: 2
Type errors: 0
Enum violations: 0

Raw output:
{
  "fault_location": {
    "component_name": "Compressor",
    "machine_name": "IBM3_C55"
  },
  "fault_symptoms": [
    "Systeem verliest He-druk",
    "druk: (uit) 192 psi",
    "druk (koud) 50 psi",
    "maakt slepend geluid",
    "niet goed",
    "koeld wel in 4uur terug, naar 13K"
  ],
  "fault_reason": [],
  "fault_measures": [],
  "resolution_status": "Resolved"
}

Parsed result:
{
  "fault_location": {
    "component_name": "Compressor",
    "machine_name": "IBM3_C55"
  },
  "fault_symptoms": [
    "Systeem verliest He-druk",
    "druk: (uit) 192 psi",
    "druk (koud) 50 psi",
    "maakt slepend geluid",
    "niet goed",
    "k

In [21]:
case_id = extract_one_loose_case(10)
evaluate_one_loose_case(case_id)

Extracting case 10/20: IBM3_C60_29_Mar_22-15-Apr-22
Saved: IBM3_C60_29_Mar_22-15-Apr-22
Raw JSON valid: True
Cleaned JSON valid: True
Case: IBM3_C60_29_Mar_22-15-Apr-22
Raw JSON valid: True
Cleaned JSON valid: True
Schema valid after cleaning: False
Neo4j ingestable: False

Missing fields: 1
Extra fields: 5
Type errors: 4
Enum violations: 1

Raw output:
{
  "fault_location": {
    "component": "Cryopomp",
    "machine": "IBM 3"
  },
  "fault_symptoms": [
    "wasmachine geluiden bij afkoelen",
    "cryopomp er net doorheen zonder lawaai (>55K)"
  ],
  "fault_reason": [
    {
      "cause": "Zeer waarschijnlijk is de cryopomp verontreinigd"
    }
  ],
  "fault_measures": [
    {
      "action": "Cryopomp gespoeld met helium volgens procedure"
    },
    {
      "action": "Adsorber vervangen"
    },
    {
      "action": "Aluminium reflector in de IBM 3 geplaatst"
    }
  ],
  "resolution_status": "Resolved"
}

Parsed result:
{
  "fault_location": {
    "component": "Cryopomp",
    "mach